# ECGClassifierV2 — Training and Export

Trains the 1-D CNN on the preprocessed MIT-BIH beat arrays, evaluates on a
held-out test split, and exports to both PyTorch checkpoint and ONNX.

**Requires** `data/all_beats.npy` and `data/all_labels.npy` from `01_data_pipeline.ipynb`.

**Outputs:**
- `results/ecg_model_v2.pth`  — PyTorch state dict
- `results/ecg_model_v2.onnx` — ONNX graph for STM32CubeAI / X-CUBE-AI
- `results/confusion_matrix.png`

### Re-running for STM32 deployment
After retraining, the ONNX export at the bottom of this notebook regenerates
`ecg_model_v2.onnx`. Load it into STM32CubeIDE via:
`ecg_classifier.ioc → X-CUBE-AI → Load Model → results/ecg_model_v2.onnx → Generate Code`

In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import onnx

## Load Data

In [ ]:
all_beats  = np.load('../data/all_beats.npy')
all_labels = np.load('../data/all_labels.npy')

X_train, X_test, y_train, y_test = train_test_split(
    all_beats, all_labels, test_size=0.2, random_state=42
)

X_train_t = torch.FloatTensor(X_train).unsqueeze(1)
X_test_t  = torch.FloatTensor(X_test).unsqueeze(1)
y_train_t = torch.LongTensor(y_train)
y_test_t  = torch.LongTensor(y_test)

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=32, shuffle=True)
test_loader  = DataLoader(TensorDataset(X_test_t,  y_test_t),  batch_size=32)

classes, counts = np.unique(all_labels, return_counts=True)
CLASS_NAMES = ['N', 'S', 'V', 'F', 'Q']

print(f"Total beats : {len(all_beats):,}")
print(f"Train       : {len(X_train):,}")
print(f"Test        : {len(X_test):,}")
print(f"\nClass distribution (full dataset):")
for c, n in zip(classes, counts):
    print(f"  {CLASS_NAMES[c]}  {n:>6,}  ({n/len(all_labels)*100:.1f}%)")

## Model — ECGClassifierV2

The key design change from v1: `AvgPool1d(kernel_size=11, stride=11)` inserted
before the fully-connected layers.  This collapses the spatial dimension from
128 × 45 → 128 × 4, reducing FC1 input from 5 760 → 512 and cutting the total
model from ~2.8 MB to 460 KB — small enough for STM32F446RE flash.

```
Without pooling:  FC1 = Linear(5760, 256)  →  737 408 params  ~2.8 MB
With    pooling:  FC1 = Linear( 512, 128)  →   65 664 params   460 KB  ✓
```

In [ ]:
class ECGClassifierV2(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1      = nn.Conv1d(1,   32,  kernel_size=5, padding=2)
        self.conv2      = nn.Conv1d(32,  64,  kernel_size=5, padding=2)
        self.conv3      = nn.Conv1d(64,  128, kernel_size=5, padding=2)
        self.pool       = nn.MaxPool1d(2)
        self.pool_adapt = nn.AvgPool1d(kernel_size=11, stride=11)  # 45 → 4, ONNX-safe
        self.relu       = nn.ReLU()
        self.dropout    = nn.Dropout(0.5)
        self.fc1        = nn.Linear(512, 128)
        self.fc2        = nn.Linear(128, 5)

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))   # 1×360  → 32×180
        x = self.pool(self.relu(self.conv2(x)))   # 32×180 → 64×90
        x = self.pool(self.relu(self.conv3(x)))   # 64×90  → 128×45
        x = self.pool_adapt(x)                    # 128×45 → 128×4
        x = x.view(x.size(0), -1)                 # → 512
        x = self.dropout(self.relu(self.fc1(x)))
        return self.fc2(x)

model = ECGClassifierV2()
total_params = sum(p.numel() for p in model.parameters())
print(f"Parameters  : {total_params:,}")
print(f"Float32 size: {total_params * 4 / 1024:.0f} KB")
print(f"INT8 size   : {total_params / 1024:.0f} KB")
print(model)

## Training

Inverse-frequency class weighting corrects for the severe class imbalance
(Normal beats make up ~83 % of MIT-BIH).

In [ ]:
class_counts  = np.bincount(y_train)
class_weights = torch.FloatTensor(1.0 / class_counts)
criterion     = nn.CrossEntropyLoss(weight=class_weights)
optimizer     = optim.Adam(model.parameters(), lr=0.001)

EPOCHS = 30
history = []

for epoch in range(EPOCHS):
    model.train()
    total_loss, correct = 0, 0
    for beats, labels in train_loader:
        optimizer.zero_grad()
        out  = model(beats)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct    += (out.argmax(1) == labels).sum().item()
    acc = correct / len(X_train) * 100
    history.append((total_loss, acc))
    print(f"Epoch {epoch+1:2d}/{EPOCHS}  loss={total_loss:7.1f}  acc={acc:.2f}%")

In [ ]:
losses, accs = zip(*history)
fig, ax1 = plt.subplots(figsize=(9, 4))
ax1.plot(losses, color='steelblue', label='Train loss')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss', color='steelblue')
ax2 = ax1.twinx()
ax2.plot(accs, color='darkorange', label='Train accuracy')
ax2.set_ylabel('Accuracy (%)', color='darkorange')
ax1.set_title('Training curve — ECGClassifierV2')
fig.legend(loc='lower right', bbox_to_anchor=(0.88, 0.15))
plt.tight_layout(); plt.show()

## Evaluation

In [ ]:
model.eval()
all_pred, all_true = [], []

with torch.no_grad():
    for beats, labels in test_loader:
        pred = model(beats).argmax(1)
        all_pred.extend(pred.numpy())
        all_true.extend(labels.numpy())

test_acc = sum(p == t for p, t in zip(all_pred, all_true)) / len(all_true) * 100
print(f"Test accuracy: {test_acc:.2f}%\n")
print(classification_report(all_true, all_pred, target_names=CLASS_NAMES))

In [ ]:
cm = confusion_matrix(all_true, all_pred)
FULL_NAMES = ['Normal', 'Supravent.', 'Ventricular', 'Fusion', 'Unclassif.']

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm, cmap='Blues')
plt.colorbar(im, ax=ax)
ax.set_xticks(range(5)); ax.set_xticklabels(FULL_NAMES, rotation=35, ha='right')
ax.set_yticks(range(5)); ax.set_yticklabels(FULL_NAMES)
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
ax.set_title(f'ECGClassifierV2 — Test Accuracy: {test_acc:.2f}%')

thresh = cm.max() / 2
for i in range(5):
    for j in range(5):
        ax.text(j, i, cm[i, j], ha='center', va='center',
                color='white' if cm[i, j] > thresh else 'black', fontsize=11)

plt.tight_layout()
plt.savefig('../results/confusion_matrix.png', dpi=120)
plt.show()
print("Saved: results/confusion_matrix.png")

## Export

Saves the PyTorch checkpoint and exports to ONNX (opset 11, static input shape).

**ONNX → STM32 workflow:**
1. Open `ecg_classifier/ecg_classifier.ioc` in STM32CubeIDE
2. X-CUBE-AI → Load Model → select `results/ecg_model_v2.onnx`
3. Analyse → Generate Code
4. Rebuild and flash

In [ ]:
torch.save(model.state_dict(), '../results/ecg_model_v2.pth')
print("Saved: results/ecg_model_v2.pth")

In [ ]:
model.eval()
dummy = torch.randn(1, 1, 360)

torch.onnx.export(
    model,
    dummy,
    '../results/ecg_model_v2.onnx',
    export_params=True,
    opset_version=11,
    dynamo=False,
    input_names=['ecg_input'],
    output_names=['class_scores'],
    dynamic_axes={
        'ecg_input':    {0: 'batch_size'},
        'class_scores': {0: 'batch_size'},
    },
)

onnx_model = onnx.load('../results/ecg_model_v2.onnx')
onnx.checker.check_model(onnx_model)
size_kb = os.path.getsize('../results/ecg_model_v2.onnx') / 1024

print(f"ONNX export : valid")
print(f"File size   : {size_kb:.1f} KB")
print(f"Input       : ecg_input   [batch x 1 x 360]  float32")
print(f"Output      : class_scores [batch x 5]        float32")
print(f"\nReady for STM32CubeAI. Load results/ecg_model_v2.onnx into X-CUBE-AI.")